In [1]:
import os
import torch

from torch.utils.data import DataLoader

from modules.configs.config import Config
from modules.utils.seed import set_seed

from modules.dataset.dataset import SpeakerDataset

from modules.models.ecapa_tdnn import ECAPATDNN
from modules.models.aam_softmax import AAMSoftmax

from modules.trainer.trainer import Trainer


def main():

    # --------------------------------------------------
    # Set Random Seed
    # --------------------------------------------------

    set_seed(Config.SEED)

    # --------------------------------------------------
    # Device
    # --------------------------------------------------

    device = Config.DEVICE

    print("=" * 60)
    print(f"Using Device : {device}")
    print("=" * 60)

    # --------------------------------------------------
    # Dataset
    # --------------------------------------------------

    train_dataset = SpeakerDataset(
        root_dir=Config.TRAIN_PATH,
        sample_rate=Config.SAMPLE_RATE,
        segment_length=Config.SEGMENT_LENGTH,
        train=True,
        augmentation=True
    )

    val_dataset = SpeakerDataset(
        root_dir=Config.VAL_PATH,
        sample_rate=Config.SAMPLE_RATE,
        segment_length=Config.SEGMENT_LENGTH,
        train=False,
        augmentation=False
    )

    # --------------------------------------------------
    # DataLoader
    # --------------------------------------------------

    train_loader = DataLoader(
        train_dataset,
        batch_size=Config.BATCH_SIZE,
        shuffle=True,
        num_workers=Config.NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=Config.NUM_WORKERS > 0,
        drop_last=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=Config.BATCH_SIZE,
        shuffle=False,
        num_workers=Config.NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=Config.NUM_WORKERS > 0,
    )

    # --------------------------------------------------
    # Number of Speakers
    # --------------------------------------------------

    num_classes = len(train_dataset.speaker_to_index)

    print(f"\nTotal Speakers : {num_classes}")
    print(f"Training Samples : {len(train_dataset)}")
    print(f"Validation Samples : {len(val_dataset)}\n")

    # --------------------------------------------------
    # Model
    # --------------------------------------------------

    model = ECAPATDNN(
        channels=Config.CHANNELS,
        scale=Config.RES2NET_SCALE,
        emb_dim=Config.EMBEDDING_SIZE
    )
    # --------------------------------------------------
    # AAM Softmax Classifier
    # --------------------------------------------------

    classifier = AAMSoftmax(
        embedding_dim=Config.EMBEDDING_SIZE,
        num_classes=num_classes,
        margin=Config.MARGIN,
        scale=Config.AAM_SCALE
    )

    # --------------------------------------------------
    # Optimizer
    # --------------------------------------------------

    optimizer = torch.optim.AdamW(
        list(model.parameters()) +
        list(classifier.parameters()),
        lr=Config.LEARNING_RATE,
        weight_decay=Config.WEIGHT_DECAY
    )

    # --------------------------------------------------
    # Learning Rate Scheduler
    # --------------------------------------------------

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=Config.NUM_EPOCHS
    )

    # --------------------------------------------------
    # Trainer
    # --------------------------------------------------

    trainer = Trainer(
        model=model,
        classifier=classifier,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        epochs=Config.NUM_EPOCHS,
        checkpoint_dir=Config.CHECKPOINT_DIR,
        mixed_precision=False 
    )

    # --------------------------------------------------
    # Resume Training
    # --------------------------------------------------

    latest_checkpoint = os.path.join(
        Config.CHECKPOINT_DIR,
        "latest_model.pt"
    )

    if os.path.exists(latest_checkpoint):

        print("\nLoading latest checkpoint...\n")

        trainer.load_checkpoint(latest_checkpoint)

    else:

        print("\nNo checkpoint found. Training from scratch...\n")

    # --------------------------------------------------
    # Start Training
    # --------------------------------------------------

    history = trainer.fit()

    print("\nTraining Completed Successfully.")

    return {
        "history": history,
        "model": model,
        "classifier": classifier
    }


if __name__ == "__main__":

    main()


Using Device : cuda
Scanning Dataset...
./data/LibriSpeech/train-clean-100
Dataset Loaded
Total Speakers : 251
Total Audio    : 28539
Scanning Dataset...
./data/LibriSpeech/dev-clean
Dataset Loaded
Total Speakers : 40
Total Audio    : 2703

Total Speakers : 251
Training Samples : 28539
Validation Samples : 2703


Loading latest checkpoint...

Checkpoint Loaded Successfully
Resume Training From Epoch 4
Starting ECAPA-TDNN Training...


Validation: 100%|██████████| 85/85 [00:09<00:00,  9.06it/s]



----------------------------------------------------------------------
Epoch 4/10
Learning Rate       : 0.00029882
Train Loss          : 0.4048
Train Accuracy      : 90.03%
Validation EER      : 9.4010%
Validation Ver. Acc : 98.45%
Best Validation EER : 9.4010%
----------------------------------------------------------------------



Validation: 100%|██████████| 85/85 [00:08<00:00,  9.61it/s]



----------------------------------------------------------------------
Epoch 5/10
Learning Rate       : 0.00029815
Train Loss          : 0.2596
Train Accuracy      : 93.68%
Validation EER      : 8.5790%
Validation Ver. Acc : 98.68%
Best Validation EER : 8.5790%
----------------------------------------------------------------------



...


----------------------------------------------------------------------
Epoch 8/10
Learning Rate       : 0.00029529
Train Loss          : 0.1894
Train Accuracy      : 95.23%
Validation EER      : 8.1446%
Validation Ver. Acc : 98.74%
Best Validation EER : 8.1446%
----------------------------------------------------------------------



Validation: 100%|██████████| 85/85 [00:08<00:00,  9.99it/s]



----------------------------------------------------------------------
Epoch 9/10
Learning Rate       : 0.00029404
Train Loss          : 0.1743
Train Accuracy      : 95.64%
Validation EER      : 8.5083%
Validation Ver. Acc : 98.67%
Best Validation EER : 8.1446%
----------------------------------------------------------------------



Validation: 100%|██████████| 85/85 [00:08<00:00,  9.73it/s]



----------------------------------------------------------------------
Epoch 10/10
Learning Rate       : 0.00029266
Train Loss          : 0.1550
Train Accuracy      : 96.25%
Validation EER      : 7.7046%
Validation Ver. Acc : 98.71%
Best Validation EER : 7.7046%
----------------------------------------------------------------------

Training Finished Successfully

Training Completed Successfully.
